## Notebook for using the CDS API to get ERA5 data

#### Run the pip install to get the latest cdsapi

In [ ]:
#!pip install cdsapi
# created .cdsapirc file in ~/ with UID and API-key per https://pypi.org/project/cdsapi/

In [1]:
import cdsapi
import requests

### This block is for grabbing a single year, entire globe <br><br> This actually errors out and says the request is too large for netcdf, apparently

In [ ]:
# c = cdsapi.Client()

# c.retrieve(
#     'reanalysis-era5-single-levels',
#     {
#         'product_type': 'reanalysis',
#         'variable': [
#             '2m_temperature', 'land_sea_mask',
#         ],
#         'year': '1980',
#         'month': [
#             '01', '02', '03',
#             '04', '05', '06',
#             '07', '08', '09',
#             '10', '11', '12',
#         ],
#         'day': [
#             '01', '02', '03',
#             '04', '05', '06',
#             '07', '08', '09',
#             '10', '11', '12',
#             '13', '14', '15',
#             '16', '17', '18',
#             '19', '20', '21',
#             '22', '23', '24',
#             '25', '26', '27',
#             '28', '29', '30',
#             '31',
#         ],
#         'time': [
#             '00:00', '01:00', '02:00',
#             '03:00', '04:00', '05:00',
#             '06:00', '07:00', '08:00',
#             '09:00', '10:00', '11:00',
#             '12:00', '13:00', '14:00',
#             '15:00', '16:00', '17:00',
#             '18:00', '19:00', '20:00',
#             '21:00', '22:00', '23:00',
#         ],
#         'format': 'netcdf',
#     },
#     'ERA5-1980-Full-T2m_LSM-download.nc')

### This block is for specific years and months, specific area lat/lon

In [ ]:
# try multiple years, same region, JJA
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature', 'land_sea_mask',
        ],
        'year': ['2020','2021','2022','2023'],
        'month': [
            '06', '07', '08',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            50, -124, 47,
            -122,
        ],
        'format': 'netcdf',
    },
    '2020-2023_JJA_PugetSound_T2m-LSM.nc')

### <br> Using the Daily Statistics Application to get daily means instead of pulling hourly data and calculating it <br> https://cds.climate.copernicus.eu/cdsapp#!/software/app-c3s-daily-era5-statistics?tab=overview <br><br> From this forum post: https://forum.ecmwf.int/t/retrieve-daily-era5-era5-land-data-using-the-cds-api/1150/1 <br><br> After running - note each month is roughly 128MB or about 1.5GB per year! <br> And, it took about 10 min total to retrieve 12 months (12 files)

In [8]:
import cdsapi
import requests
 
# CDS API script to use CDS service to retrieve daily ERA5* variables and iterate over
# all months in the specified years.
 
# Requires:
# 1) the CDS API to be installed and working on your system
# 2) You have agreed to the ERA5 Licence (via the CDS web page)
# 3) Selection of required variable, daily statistic, etc
 
# Output:
# 1) separate netCDF file for chosen daily statistic/variable for each month
 
c = cdsapi.Client()#timeout=300)
 
# Uncomment years as required
 
years =  [
#    '1960',
#    '1961','1962','1963','1964'
 #   '1965',
#    '1966','1967','1968','1969'
#            '1970'
#    '1971','1972','1973','1974','1975'
#    '1976','1977','1978','1979',
#            '1980', 
#    '1981','1982', '1983', '1984',
#            '1985',
'1986', '1987','1988', '1989', 
#    '1990'#,
#            '1991', '1992', '1993',
#            '1994', '1995', '1996',
#            '1997', '1998', '1999'#,
#            '2000', 
#    '2001', '2002',
#            '2003', '2004', '2005',
#            '2006', '2007', '2008',
#            '2009', 
#   '2010', '2011',
#            '2012', '2013', '2014',
#            '2015', '2016', '2017',
#            '2018', '2019', '2020'
#            '2021',
#             '2022'
#    '2023'
]
 
 
# Retrieve all months for a given year.
 
months = ['01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12']
 
# For valid keywords, see Table 2 of:
# https://datastore.copernicus-climate.eu/documents/app-c3s-daily-era5-statistics/C3S_Application-Documentation_ERA5-daily-statistics-v2.pdf
 
# select your variable; name must be a valid ERA5 CDS API name.
var = "2m_temperature"
 
# Select the required statistic, valid names given in link above
stat = "daily_mean"
 
# Loop over years and months
 
for yr in years:
    for mn in months:
        result = c.service(
        "tool.toolbox.orchestrator.workflow",
        params={
             "realm": "user-apps",
             "project": "app-c3s-daily-era5-statistics",
             "version": "master",
             "kwargs": {
                 "dataset": "reanalysis-era5-single-levels",
                 "product_type": "reanalysis",
                 "variable": var,
                 "statistic": stat,
                 "year": yr,
                 "month": mn,
                 "time_zone": "UTC+00:0",
                 "frequency": "1-hourly",
#
# Users can change the output grid resolution and selected area
#
#                "grid": "1.0/1.0",
#                "area":{"lat": [10, 60], "lon": [65, 140]}
 
                 },
        "workflow_name": "application"
        })
         
# set name of output file for each month (statistic, variable, year, month     
 
        file_name = "../../../Data/ERA5-global/Baseline/" + yr + "/download_" + stat + "_" + var + "_" + yr + "_" + mn + ".nc"
         
        location=result[0]['location']
        res = requests.get(location, stream = True)
        print("Writing data to " + file_name)
        with open(file_name,'wb') as fh:
            for r in res.iter_content(chunk_size = 1024):
                fh.write(r)
        fh.close()

2024-06-26 13:56:24,386 INFO Welcome to the CDS
2024-06-26 13:56:24,387 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-4990520755e94cc6ac288896e9b756f4
2024-06-26 13:56:24,668 INFO Request is queued
2024-06-26 13:56:25,923 INFO Request is running
2024-06-26 13:56:58,920 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1986/download_daily_mean_2m_temperature_1986_01.nc


2024-06-26 13:57:23,408 INFO Welcome to the CDS
2024-06-26 13:57:23,409 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-f7880c4b38bb4eada51363db73abd8bf
2024-06-26 13:57:23,652 INFO Request is queued
2024-06-26 13:57:24,836 INFO Request is running
2024-06-26 13:57:28,960 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1986/download_daily_mean_2m_temperature_1986_02.nc


2024-06-26 13:58:28,450 INFO Welcome to the CDS
2024-06-26 13:58:28,451 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-893526da34004672bb82743b464288a0
2024-06-26 13:58:28,817 INFO Request is queued
2024-06-26 13:58:30,001 INFO Request is running
2024-06-26 13:58:34,125 INFO Request is completed


...

Writing data to ../../../Data/ERA5-global/Baseline/1989/download_daily_mean_2m_temperature_1989_10.nc


2024-06-26 14:37:41,141 INFO Welcome to the CDS
2024-06-26 14:37:41,142 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-8d70203b30694f5b84cb5a9e7636ed09
2024-06-26 14:37:41,334 INFO Request is queued
2024-06-26 14:37:42,512 INFO Request is running
2024-06-26 14:37:46,626 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1989/download_daily_mean_2m_temperature_1989_11.nc


2024-06-26 14:38:10,081 INFO Welcome to the CDS
2024-06-26 14:38:10,082 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-bc306e13461245409c41afed297d0ba1
2024-06-26 14:38:10,283 INFO Request is queued
2024-06-26 14:38:11,465 INFO Request is running
2024-06-26 14:38:32,135 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1989/download_daily_mean_2m_temperature_1989_12.nc


## 

{'r': 327,
 'fh': 168,
 'location': 163,
 'months': 152,
 'open': 152,
 'file_name': 94,
 'result': 88,
 'years': 64,
 'var': 63,
 'c': 56,
 'res': 56,
 'yr': 53,
 'mn': 51}

In [9]:
!ls -al "../../../Data/ERA5-global/Baseline/1989/"

total 2961552
drwxr-xr-x@ 14 tedscott  staff        448 Jun 26 14:39 .
drwxr-xr-x@ 36 tedscott  staff       1152 Jun 26 10:24 ..
-rw-r--r--@  1 tedscott  staff  128778534 Jun 26 14:29 download_daily_mean_2m_temperature_1989_01.nc
-rw-r--r--@  1 tedscott  staff  116319629 Jun 26 14:30 download_daily_mean_2m_temperature_1989_02.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jun 26 14:31 download_daily_mean_2m_temperature_1989_03.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jun 26 14:31 download_daily_mean_2m_temperature_1989_04.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jun 26 14:32 download_daily_mean_2m_temperature_1989_05.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jun 26 14:33 download_daily_mean_2m_temperature_1989_06.nc
-rw-r--r--@  1 tedscott  staff  128778533 Jun 26 14:36 download_daily_mean_2m_temperature_1989_07.nc
-rw-r--r--@  1 tedscott  staff  128778531 Jun 26 14:36 download_daily_mean_2m_temperature_1989_08.nc
-rw-r--r--@  1 tedscott  staff  124625565 Jun 26 14:37 download